In [1]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import json
from datetime import datetime, timezone, timedelta
import pickle
import os
from src.utils.ficheros import guardarExcel, guardarExcelMulti
from pathlib import Path
from src.utils.util import loadEstaciones,loadEstacionSinCTC
GRAYLOG_URL = "http://silog.solvan.comun.adif"
USER = "E942433"
PASSWORD = "Xiaohuanghua5"

# def search_logs(query, from_date, to_date, stream_id=None):
#     url = f"{GRAYLOG_URL}/api/search/universal/absolute"
    
#     params = {
#         "query": query,
#         "from": from_date,
#         "to": to_date,
#         "offset": 0,
#         "fields": "timestamp,source,message,level"
#     }
    
#     # Añadir filtro de stream si se especifica
#     if stream_id:
#         params["filter"] = f"streams:{stream_id}"
    
#     headers = {"Accept": "application/json"}
    
#     response = requests.get(
#         url,
#         params=params,
#         auth=HTTPBasicAuth(USER, PASSWORD),
#         headers=headers
#     )
    
#     response.raise_for_status()
#     return response.json()


# Ver streams disponibles
def get_streams():
    url = f"{GRAYLOG_URL}/api/streams"
    response = requests.get(
        url,
        auth=HTTPBasicAuth(USER, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    streams = response.json()["streams"]
    for s in streams:
        print(f"ID: {s['id']} | Nombre: {s['title']}")
    return streams



In [2]:
import time
def graylog_get(url, params, headers, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(
                url, params=params,
                auth=HTTPBasicAuth(USER, PASSWORD),
                headers=headers, timeout=30
            )
            if r.status_code == 500:
                raise requests.exceptions.HTTPError("500", response=r)
            r.raise_for_status()
            return r.json()

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                raise  # Propagar 500 sin reintentar — lo gestiona fetch_window
            raise

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = 2 ** attempt
            print(f"\n⏱️  Error red — reintento {attempt+1}/{max_retries} en {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"❌ Fallaron {max_retries} reintentos")


def fetch_window(query, from_str, to_str, stream_id, headers,
                 batch_size=5000, _depth=0):
    """
    Descarga una ventana de tiempo. Si encuentra error 500 por offset alto,
    divide la ventana en 2 mitades y las descarga recursivamente.
    Máximo 8 niveles de recursión (ventana mínima ~1s).
    """
    if _depth > 8:
        print(f"\n⛔ Ventana demasiado densa incluso dividida: {from_str} → {to_str}")
        return []

    messages = []
    offset    = 0

    while True:
        params = {
            "query": query, "from": from_str, "to": to_str,
            "limit": batch_size, "offset": offset,
            "fields": "timestamp,source,message,level,contentType"
        }
        if stream_id:
            params["filter"] = f"streams:{stream_id}"

        try:
            data  = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
            batch = data.get("messages", [])
            total = data.get("total_results", 0)
            messages.extend(batch)
            offset += len(batch)
            print(f"  {from_str} → {to_str} | {offset}/{total}", end="\r")
            if offset >= total or not batch:
                break

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                # Demasiados resultados con offset alto → dividir ventana en 2
                t_from = datetime.fromisoformat(from_str.replace("Z", "+00:00"))
                t_to   = datetime.fromisoformat(to_str.replace("Z", "+00:00"))
                mid    = t_from + (t_to - t_from) / 2
                mid_str = mid.strftime("%Y-%m-%dT%H:%M:%S.000Z")

                print(f"\n✂️  Dividiendo [{from_str} → {to_str}] (offset={offset}, depth={_depth})")

                # Descartar mensajes parciales de esta ventana y rehacer por mitades
                left  = fetch_window(query, from_str, mid_str, stream_id, headers,
                                     batch_size, _depth + 1)
                right = fetch_window(query, mid_str, to_str, stream_id, headers,
                                     batch_size, _depth + 1)
                return messages[:offset - len(batch)] + left + right
            raise

    return messages


def get_total_results(query, from_str, to_str, stream_id, headers):
    params = {"query": query, "from": from_str, "to": to_str,
              "limit": 1, "offset": 0, "fields": "timestamp"}
    if stream_id:
        params["filter"] = f"streams:{stream_id}"
    data = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
    return data.get("total_results", 0)


def search_logs(query, from_date, to_date, stream_id=None, limit=None,
                max_per_window=5000, checkpoint_file="checkpoint.pkl"):
    headers      = {"Accept": "application/json"}
    all_messages = []
    range_start  = datetime.fromisoformat(from_date.replace("Z", "+00:00"))
    range_end    = datetime.fromisoformat(to_date.replace("Z", "+00:00"))
    total_seconds = (range_end - range_start).total_seconds()

    # Reanudar desde checkpoint
    resume_from = range_start
    if os.path.exists(checkpoint_file):
        print(f"♻️  Reanudando desde checkpoint...")
        with open(checkpoint_file, "rb") as f:
            ckpt = pickle.load(f)
        all_messages = ckpt["messages"]
        resume_from  = ckpt["last_window_end"]
        print(f"   {len(all_messages):,} msgs ya descargados, continuando desde {resume_from}\n")

    # Calcular ventana óptima
    print("🔍 Calculando total de mensajes...")
    total_global = get_total_results(
        query,
        range_start.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        range_end.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        stream_id, headers
    )
    print(f"📊 Total: {total_global:,}")

    if total_global == 0:
        return []

    density        = total_global / total_seconds
    window_seconds = max(int((max_per_window / density) * 0.85), 5)
    remaining      = (range_end - resume_from).total_seconds()
    print(f"⚙️  Ventana: {window_seconds}s | Estimadas: {int(remaining/window_seconds)+1}\n")

    current      = resume_from
    window_count = 0

    while current < range_end:
        window_end = min(current + timedelta(seconds=window_seconds), range_end)
        from_str   = current.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        to_str     = window_end.strftime("%Y-%m-%dT%H:%M:%S.000Z")

        try:
            batch = fetch_window(query, from_str, to_str, stream_id, headers)
            all_messages.extend(batch)
            window_count += 1
            print(f"✅ {from_str} → {to_str} | +{len(batch):,} | Total: {len(all_messages):,}")

        except RuntimeError as e:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": current}, f)
            print(f"\n💾 Guardado emergencia: {len(all_messages):,} msgs")
            raise e

        # Checkpoint cada 50 ventanas
        if window_count % 50 == 0:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": window_end}, f)
            print(f"💾 Checkpoint: {len(all_messages):,} msgs")

        current = window_end

        if limit and len(all_messages) >= limit:
            all_messages = all_messages[:limit]
            print(f"\n🛑 Límite alcanzado: {limit:,} msgs")
            break

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"\n✅ Descarga completa: {len(all_messages):,} mensajes")
    return all_messages


def search_logs_to_dataframe(query, from_date, to_date, stream_id=None,
                              limit=None, checkpoint_file="checkpoint.pkl"):
    messages = search_logs(query, from_date, to_date, stream_id, limit,
                           checkpoint_file=checkpoint_file)
    if not messages:
        return pd.DataFrame()
    rows = [msg.get("message", msg) for msg in messages]
    return pd.DataFrame(rows)

In [3]:
data = search_logs(
    query="contentType:(Block OR Signal OR TrackCircuit OR LevelCrossing) AND ctc:MAC",
    from_date="2026-03-24T00:00:00.000Z",
    to_date="2026-03-25T00:00:00.000Z",
    stream_id="68fb73bc6456d79315e70710",
    limit=None  )

🔍 Calculando total de mensajes...
📊 Total: 1,129,169
⚙️  Ventana: 325s | Estimadas: 266

✅ 2026-03-24T00:00:00.000Z → 2026-03-24T00:05:25.000Z | +1,402 | Total: 1,402
✅ 2026-03-24T00:05:25.000Z → 2026-03-24T00:10:50.000Z | +1,288 | Total: 2,690
✅ 2026-03-24T00:10:50.000Z → 2026-03-24T00:16:15.000Z | +1,501 | Total: 4,191
✅ 2026-03-24T00:16:15.000Z → 2026-03-24T00:21:40.000Z | +1,329 | Total: 5,520
✅ 2026-03-24T00:21:40.000Z → 2026-03-24T00:27:05.000Z | +1,248 | Total: 6,768
✅ 2026-03-24T00:27:05.000Z → 2026-03-24T00:32:30.000Z | +1,131 | Total: 7,899
✅ 2026-03-24T00:32:30.000Z → 2026-03-24T00:37:55.000Z | +831 | Total: 8,730
✅ 2026-03-24T00:37:55.000Z → 2026-03-24T00:43:20.000Z | +1,127 | Total: 9,857
✅ 2026-03-24T00:43:20.000Z → 2026-03-24T00:48:45.000Z | +1,106 | Total: 10,963
✅ 2026-03-24T00:48:45.000Z → 2026-03-24T00:54:10.000Z | +1,001 | Total: 11,964
✅ 2026-03-24T00:54:10.000Z → 2026-03-24T00:59:35.000Z | +898 | Total: 12,862
✅ 2026-03-24T00:59:35.000Z → 2026-03-24T01:05:00.000Z 

In [6]:
lista_messages = [item['message']['message'] for item in data]

In [7]:
df = pd.json_normalize(lista_messages)

In [8]:
dict_list = [json.loads(x) for x in lista_messages]

In [9]:
df = pd.json_normalize(dict_list)

In [11]:
df.head(4)

,version,header.crc,header.ctc,header.timestampMSG,header.timeStampCTC,header.originSystem,header.sequenceNumber,header.ContentType,messageType.ChangeState.trackCircuit.element.ctc,messageType.ChangeState.trackCircuit.element.interlock,...,messageType.ChangeState.levelCrossing.element.ctc,messageType.ChangeState.levelCrossing.element.interlock,messageType.ChangeState.levelCrossing.element.name,messageType.ChangeState.levelCrossing.state.upToDate,messageType.ChangeState.levelCrossing.state.barrier,messageType.ChangeState.levelCrossing.state.LCBlownLamps.lamp1,messageType.ChangeState.levelCrossing.state.LCBlownLamps.lamp2,messageType.ChangeState.levelCrossing.state.interlocked,messageType.ChangeState.levelCrossing.state.manual,messageType.ChangeState.levelCrossing.state.alarm
0,1.3.2,3325754447,MAC,2026-03-24 00:05:18.688,1774310718000,MSE,20826,TrackCircuit,MAC,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.3.2,927607276,MAC,2026-03-24 00:05:22.694,1774310722000,MSE,20840,TrackCircuit,MAC,AZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.3.2,612162956,MAC,2026-03-24 00:05:20.542,1774310720000,MSE,20834,TrackCircuit,MAC,AZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.3.2,3934136837,MAC,2026-03-24 00:05:22.676,1774310722000,MSE,20839,TrackCircuit,MAC,AZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df['header.timestampMSG'] = pd.to_datetime(df['header.timestampMSG'], unit='ms')

In [51]:
df['header.timeStampCTC'] = pd.to_datetime(df['header.timeStampCTC'], unit='ms')

In [52]:
df_filter = df[["header.ctc","header.ContentType"]].copy()

In [53]:
df_filter["element.interlock"] = (
    df["messageType.ChangeState.trackCircuit.element.interlock"]
        .combine_first(df["messageType.ChangeState.block.element.interlock"])
        .combine_first(df["messageType.ChangeState.signal.element.interlock"])
        .combine_first(df["messageType.ChangeState.levelCrossing.element.interlock"])
)

In [54]:
df_filter["element.name"] = (
    df["messageType.ChangeState.trackCircuit.element.name"]
        .combine_first(df["messageType.ChangeState.block.element.name"])
        .combine_first(df["messageType.ChangeState.signal.element.name"])
        .combine_first(df["messageType.ChangeState.levelCrossing.element.name"])
)

In [55]:
df_filter["element.type"] = (
    df["messageType.ChangeState.trackCircuit.element.type"]
        .combine_first(df["messageType.ChangeState.signal.element.type"])
        .combine_first(df["messageType.ChangeState.signal.element.type"])
        #.combine_first(df["messageType.ChangeState.levelCrossing.element.type"])
)

In [59]:
data

[{'highlight_ranges': {},
  'message': {'_id': '28d39f30-2715-11f1-89cc-00505683230e',
   'source': 'VILRELC016',
   'message': '{"version":"1.3.2","header":{"crc":3325754447,"ctc":"MAC","timestampMSG":1774310718688,"timeStampCTC":1774310718000,"originSystem":"MSE","sequenceNumber":20826,"ContentType":"TrackCircuit"},"messageType":{"ChangeState":{"trackCircuit":{"element":{"ctc":"MAC","interlock":"AU","name":"A9","type":"Switch"},"state":{"upToDate":true,"trainOccupation":false,"interlockedRoute":{"value":true,"routeType":"Itinerario"},"failure":"NOT","blocked":{"value":false,"type":""},"dial":false,"switch":{"checkedPosition":{"value":true,"position":"izquierda"},"commandedPosition":{"value":false,"position":"izquierda"},"interlocked":true,"individualBlock":true,"heeloperated":false,"insufficientGauge":{"value":false,"position":""},"authorizedMaintenance":false,"movementRequired":false}},"trains":{"train":[]}}}}}',
   'contentType': 'TrackCircuit',
   'timestamp': '2026-03-24T00:05:24

In [15]:
df_filter["element.interlock"].unique()

array(['RE', 'LH', 'CW', 'SS', 'MR', 'BK', 'FI', 'TB', 'LK', 'LQ', 'BX',
       'SC', 'RL', 'VR', 'BV', 'MM', 'TG', 'TC', 'AT', 'LX', 'MJ', 'MY',
       'EV', 'AC', 'VL', 'BR', 'SH', 'BO', 'BN', 'BT', 'MD', 'RC', 'TT',
       'XC', 'BS', 'CO', 'X9', 'RO', 'GC', 'MA', 'BE', 'MQ', 'CQ', 'SD',
       'TE', 'VM', 'TR', 'EP', 'BY', 'MO', 'GM', 'LG', 'RX', 'BQ', 'BL',
       'BI', 'BH', 'BD', 'BC', 'BA', 'AM', 'SM', 'VF', 'EN', 'ML', 'VU',
       'VI', 'VC', 'TO', 'SV', 'SX', 'ST', 'SQ', 'SO', 'SN', 'SL', 'RV',
       'RU', 'RT', 'RP', 'RR', 'RF', 'PU', 'PN', 'PE', 'PD', 'PA', 'GA',
       'CF', 'MT', 'MS', 'MN', 'MK', 'MG', 'ME', 'LR', 'CV', 'CM', 'EM',
       'FS', 'LN', 'CJ', 'CN', 'CY', 'CL', 'GV', 'CT', 'CU', 'CA', 'CS',
       'GO'], dtype=object)

In [16]:
# MAC = df_filter[df_filter["element.interlock"] == "BR"].copy()
MAC = df_filter.copy()

In [17]:
MAC.drop_duplicates(keep = "first")

,header.ctc,header.ContentType,element.interlock,element.name,element.type
0,BCN,TrackCircuit,RE,EP4,TrackCircuit
1,BCN,TrackCircuit,RE,175,TrackCircuit
2,BCN,TrackCircuit,LH,A39,TrackCircuit
3,BCN,TrackCircuit,LH,A57,Switch
4,BCN,TrackCircuit,LH,A39,Switch
...,...,...,...,...,...
691241,BCN,TrackCircuit,GM,A15,Switch
691242,BCN,TrackCircuit,GM,A17,Switch
691655,BCN,Signal,CF,R3,Generic
694297,BCN,Signal,GC,S2/1,Generic


In [18]:
MAC

,header.ctc,header.ContentType,element.interlock,element.name,element.type
0,BCN,TrackCircuit,RE,EP4,TrackCircuit
1,BCN,TrackCircuit,RE,175,TrackCircuit
2,BCN,TrackCircuit,LH,A39,TrackCircuit
3,BCN,TrackCircuit,LH,A57,Switch
4,BCN,TrackCircuit,LH,A39,Switch
...,...,...,...,...,...
694903,BCN,TrackCircuit,GC,A16,Switch
694904,BCN,TrackCircuit,GC,A8,Switch
694905,BCN,TrackCircuit,GC,A14,Switch
694906,BCN,TrackCircuit,VR,A7,TrackCircuit


In [19]:
import requests
def getElments(interlock,ctc):

    url = "http://topo.rail.api.elcano.operaciones.adif/msetopo/download/filesInterlock"

    payload = {
        "interlock": interlock,
         "ctc": ctc,
        #  "interlockName": "CHAMARTIN"

    }
    response = requests.post(url, json=payload)

    if response.status_code == 200:
        content_type = response.headers.get("Content-Type", "")
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None
    data= response.json()
    config_string = data["topoResource"]["configFileJson"]
    config_object = json.loads(config_string)
    topo_info = {
    "id": config_object["id"],
    "IdCatalogo": config_object["version"]["mieCatalogue"]["id"],
    "VersionCatalogo": config_object["version"]["mieCatalogue"]["version"],
    "IdCTC": config_object["info"]["ctc"]["id"],
    "Mnemónico": config_object["info"]["interlocking"],
    "NombreEnclavamiento": config_object["info"]["name"],
    "Dependencias": [d["code"] for d in config_object["info"]["dependencies"]]
    }

    data_rows = []

    for e in config_object["viewCtc"]["elements"]:
        row = topo_info.copy() 
        row.update({
            "ElementoID": e["id"],
            "NombreElemento": e["name"],
            "TipoElemento": e["type"],
            "SubtipoElemento": e["subtype"],
            "NombreCircuito":e["trackCircuitName"],
    
        })
        data_rows.append(row)

    df_topo = pd.DataFrame(data_rows)
    return df_topo


    
        

In [20]:
CTC = MAC["header.ctc"].unique()[0]

In [21]:
interlocking = MAC["element.interlock"].unique()

In [22]:
import numpy as np
dfs = []
if CTC == "BCN":
    interlocking = interlocking[interlocking != "XC"] 
elif CTC == "MAC":
        interlocking = interlocking[
        (interlocking != "DU") & 
        (interlocking != "GO") & 
        (interlocking != "AI")
    ]
for interlock in interlocking:
    try:
        
        df_topo = getElments(interlock, CTC)

        if df_topo is None or df_topo.empty:
            print(f"{interlock} returned empty")

        else:
            dfs.append(df_topo)

        time.sleep(2)

    except Exception as e:
        print(f"{interlock} failed: {e}")

df_topos = pd.concat(dfs, ignore_index=True)


In [23]:
def to_camel_case(s):
    if pd.isna(s):
        return ""
    parts = str(s).split()
    return parts[0].lower() + ''.join(p.capitalize() for p in parts[1:])

MAC["ElementoID"] = (
    MAC["header.ctc"] + "." +
    MAC["element.interlock"] + "." +
    MAC["header.ContentType"].apply(to_camel_case) + "." +
    MAC["element.name"]
)

In [24]:
dependencias = df_topos[["Mnemónico","NombreEnclavamiento"]].copy()

In [25]:
dependencias.drop_duplicates(subset="Mnemónico",inplace= True)

In [26]:
MAC = pd.merge(
    MAC,
    dependencias,
    left_on = "element.interlock",
    right_on = "Mnemónico",
    how = "left"
)

In [27]:
MAC

,header.ctc,header.ContentType,element.interlock,element.name,element.type,ElementoID,Mnemónico,NombreEnclavamiento
0,BCN,TrackCircuit,RE,EP4,TrackCircuit,BCN.RE.trackcircuit.EP4,RE,RE-Reus
1,BCN,TrackCircuit,RE,175,TrackCircuit,BCN.RE.trackcircuit.175,RE,RE-Reus
2,BCN,TrackCircuit,LH,A39,TrackCircuit,BCN.LH.trackcircuit.A39,LH,LH-L'Hospitalet-De-Llobregat
3,BCN,TrackCircuit,LH,A57,Switch,BCN.LH.trackcircuit.A57,LH,LH-L'Hospitalet-De-Llobregat
4,BCN,TrackCircuit,LH,A39,Switch,BCN.LH.trackcircuit.A39,LH,LH-L'Hospitalet-De-Llobregat
...,...,...,...,...,...,...,...,...
694903,BCN,TrackCircuit,GC,A16,Switch,BCN.GC.trackcircuit.A16,GC,GC-Granollers-Centre
694904,BCN,TrackCircuit,GC,A8,Switch,BCN.GC.trackcircuit.A8,GC,GC-Granollers-Centre
694905,BCN,TrackCircuit,GC,A14,Switch,BCN.GC.trackcircuit.A14,GC,GC-Granollers-Centre
694906,BCN,TrackCircuit,VR,A7,TrackCircuit,BCN.VR.trackcircuit.A7,VR,VR-Vilassar-De-Mar


In [28]:
# df_topos["Mnemónico_Depedencias"] = df_topos["Mnemónico"] + df_topos["Dependencias"].astype(str)

In [29]:
# MAC["Mnemónico_Depedencias"] = MAC["element.interlock"] + MAC["Dependencias"].astype(str)

In [30]:
topos_enclavamientos = {i: g for i, g in df_topos.groupby('NombreEnclavamiento')}


In [31]:
enclavamientos_recibidos =  {i: g for i, g in MAC.groupby('NombreEnclavamiento')}

In [32]:
Topo_sin_recibir = {}
recibir_sin_topo = {}

for i in topos_enclavamientos.keys() & enclavamientos_recibidos.keys():
    df1_filtrado = topos_enclavamientos[i][
        (~topos_enclavamientos[i]['ElementoID'].str.lower().str.contains('alarm|undefined|operationcontrol', na=False)) &
        (topos_enclavamientos[i]['SubtipoElemento'] != 'trackCircuitNotSignalized')
    ]

    # Crear diccionarios para mapear minúsculas → original
    nombres1_dict = {x.lower(): x for x in df1_filtrado['ElementoID']}
    nombres1_pre_set = set(x.lower() for x in df1_filtrado['NombreCircuito'])
    nombres2_dict = {x.lower(): x for x in enclavamientos_recibidos[i]['ElementoID']}

    diff1 = set(nombres1_dict.keys()) - set(nombres2_dict.keys())
    diff2 = set(nombres2_dict.keys()) - set(nombres1_dict.keys())

    if diff1:
        # Guardamos los nombres originales
        Topo_sin_recibir[i] = [nombres1_dict[n] for n in diff1]

    diff2_filtrado = []
    for n in diff2:
        if n not in diff1 and n not in nombres1_pre_set:
            diff2_filtrado.append(nombres2_dict[n])
    if diff2_filtrado:
        recibir_sin_topo[i] = diff2_filtrado

In [33]:
Topo_sin_recibir 

{'BQ-Barcelona-Sant-Andreu': ['BCN.BQ.signal.S1/4A',
  'BCN.BQ.signal.S1/2A',
  'BCN.BQ.signal.E6',
  'BCN.BQ.signal.S3',
  'BCN.BQ.signal.M2',
  'BCN.BQ.signal.S2/1',
  'BCN.BQ.signal.M4',
  'BCN.BQ.block.BS1',
  'BCN.BQ.signal.E8',
  'BCN.BQ.signal.S1/2',
  'BCN.BQ.signal.R4',
  'BCN.BQ.signal.S2/D',
  'BCN.BQ.localOperation.M.L.2',
  'BCN.BQ.signal.S1/D',
  'BCN.BQ.localOperation.M.L.1',
  'BCN.BQ.signal.E5',
  'BCN.BQ.signal.R5'],
 'RR-Riera-De-Rubi': ['BCN.RR.trackCircuit.CH1'],
 'VU-Vilanova-I-La-Geltru': ['BCN.VU.signal.6316-1',
  'BCN.VU.signal.6316-2',
  'BCN.VU.signal.6333-1',
  'BCN.VU.signal.6333-2'],
 'TG-Tarragona': ['BCN.TG.localOperation.M.L.3',
  'BCN.TG.trackCircuit.CH8',
  'BCN.TG.trackCircuit.CH2',
  'BCN.TG.block.TC1',
  'BCN.TG.trackCircuit.CH1',
  'BCN.TG.localOperation.M.L.1',
  'BCN.TG.localOperation.M.L.2',
  'BCN.TG.trackCircuit.CH3',
  'BCN.TG.trackCircuit.E3M',
  'BCN.TG.trackCircuit.CH6',
  'BCN.TG.signal.EP1',
  'BCN.TG.trackCircuit.S1',
  'BCN.TG.trackCi

In [34]:
import colorsys
import random as _random

def random_pastel(seed: str):
    rng = _random.Random(seed)
    h = rng.random()
    s = 0.45 + rng.random() * 0.2
    l = 0.78 + rng.random() * 0.08
 
    def hls_to_hex(h, l, s):
        r, g, b = colorsys.hls_to_rgb(h, l, s)
        return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))
 
    bg     = hls_to_hex(h, l,        s)
    border = hls_to_hex(h, l - 0.25, s + 0.1)
    text   = hls_to_hex(h, l - 0.42, s + 0.1)
    return bg, border, text
 
 
def build_color_map(*dicts):
    all_keys = set()
    for d in dicts:
        all_keys.update(d.keys())
    return {k: random_pastel(k) for k in all_keys}

In [35]:
# def build_html(dict_a: dict, dict_b: dict,
#                label_a='ELEMENTOS EN LAS TOPOLOGÍAS DE LAS QUE NO RECIBE INFORMACIÓN',
#                label_b='ELEMENTOS QUE RECIBIMOS INFORMACIÓN PERO NO ESTAN BIEN INCLUIDOS EN LA TOPOLOGÍAS',
#                output_title='Panel Diccionarios BCN') -> str:
 
#     colors = build_color_map(dict_a, dict_b)
#     all_keys = sorted(set(dict_a) | set(dict_b))
 
#     colors_json   = json.dumps({k: list(v) for k, v in colors.items()}, ensure_ascii=False)
#     dict_a_json   = json.dumps(dict_a, ensure_ascii=False)
#     dict_b_json   = json.dumps(dict_b, ensure_ascii=False)
#     all_keys_json = json.dumps(all_keys, ensure_ascii=False)
 
#     total_a = sum(len(v) for v in dict_a.values())
#     total_b = sum(len(v) for v in dict_b.values())

 
#     html = f"""<!DOCTYPE html>
# <html lang="es">
# <head>
# <meta charset="UTF-8">
# <meta name="viewport" content="width=device-width, initial-scale=1.0">
# <title>{output_title}</title>
# <style>
# * {{ box-sizing: border-box; margin: 0; padding: 0; }}
# body {{ font-family: system-ui, sans-serif; background: #f4f4f2; color: #1a1a1a; }}
 
# header {{ background: #1a1a2e; color: #fff; padding: 1.25rem 2rem; }}
# header h1 {{ font-size: 1.3rem; font-weight: 500; }}
# header p  {{ font-size: 0.85rem; opacity: 0.55; margin-top: 3px; }}
 
# .stats {{ display: flex; gap: 12px; flex-wrap: wrap; padding: 1.25rem 2rem 0; }}
# .stat {{ background: #fff; border-radius: 8px; padding: .75rem 1.25rem;
#          border: 1px solid #e8e8e8; min-width: 120px; }}
# .stat .val {{ font-size: 1.6rem; font-weight: 600; }}
# .stat .lbl {{ font-size: 11px; color: #888; margin-top: 2px; }}
 
# .keys-section {{ padding: 1.25rem 2rem 0; }}
# .keys-section h2 {{ font-size: 11px; text-transform: uppercase; letter-spacing: .07em;
#                     color: #aaa; margin-bottom: 10px; }}
# .keys-grid {{ display: flex; flex-wrap: wrap; gap: 10px; }}
 
# .key-card {{ padding: 10px 22px; border-radius: 10px; border: 2px solid transparent;
#              cursor: pointer; font-size: 14px; font-weight: 700;
#              transition: transform .12s, box-shadow .12s; user-select: none; }}
# .key-card:hover {{ transform: translateY(-2px); box-shadow: 0 4px 14px rgba(0,0,0,.12); }}
# .key-card.active {{ box-shadow: 0 0 0 3px rgba(0,0,0,.18); }}
# .key-card .kc-counts {{ font-size: 11px; font-weight: 400; margin-top: 3px; opacity: .7; }}
 
# .detail {{ padding: 1.25rem 2rem 2rem; }}
# .detail-placeholder {{ display: flex; align-items: center; justify-content: center;
#                         height: 140px; color: #ccc; font-size: 14px; }}
 
# .detail-header {{ display: flex; align-items: center; gap: 10px; margin-bottom: 1.25rem;
#                   padding-bottom: 10px; border-bottom: 1px solid #e8e8e8; }}
# .detail-header h2 {{ font-size: 1.4rem; font-weight: 700; }}
# .detail-badge {{ font-size: 11px; padding: 3px 10px; border-radius: 99px; font-weight: 500; }}
 
# .dict-panels {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
# @media (max-width: 680px) {{ .dict-panels {{ grid-template-columns: 1fr; }} }}
 
# .dict-panel {{ background: #fff; border-radius: 10px; border: 1px solid #e8e8e8;
#                padding: 14px 16px; }}
# .dict-panel h3 {{ font-size: 12px; font-weight: 600; text-transform: uppercase;
#                   letter-spacing: .06em; margin-bottom: 10px; padding-bottom: 8px;
#                   border-bottom: 1px solid #f0f0f0; }}
# .dict-panel.a h3 {{ color: #2563eb; }}
# .dict-panel.b h3 {{ color: #059669; }}
# .dict-panel.empty-panel {{ color: #bbb; font-size: 13px; display: flex;
#                             align-items: center; justify-content: center; min-height: 80px; }}
 
# .type-block {{ margin-bottom: 10px; }}
# .type-lbl {{ font-size: 10px; text-transform: uppercase; letter-spacing: .06em;
#              color: #bbb; margin-bottom: 4px; }}
# .pills {{ display: flex; flex-wrap: wrap; gap: 5px; }}
# .pill {{ font-size: 12px; padding: 3px 11px; border-radius: 99px;
#          background: #f5f5f3; border: 1px solid #e4e4e0; cursor: default; }}
# </style>
# </head>
# <body>
 
# <header>
#   <h1>{output_title}</h1>
#   <p>Haz clic en una clave para ver los detalles</p>
# </header>
 
# <div class="stats">
#   <div class="stat"><div class="val">{len(all_keys)}</div><div class="lbl">Enclavamientos totales</div></div>
#   <div class="stat"><div class="val">{total_a}</div><div class="lbl">{label_a}</div></div>
#   <div class="stat"><div class="val">{total_b}</div><div class="lbl">{label_b}</div></div>
# </div>
 
# <div class="keys-section">
#   <h2>Claves — selecciona una</h2>
#   <div class="keys-grid" id="keysGrid"></div>
# </div>
 
# <div class="detail" id="detail">
#   <div class="detail-placeholder">Selecciona una clave para ver su detalle</div>
# </div>
 
# <script>
# const DICT_A    = {dict_a_json};
# const DICT_B    = {dict_b_json};
# const LABEL_A   = "{label_a}";
# const LABEL_B   = "{label_b}";
# const ALL_KEYS  = {all_keys_json};
# const COLORS    = {colors_json};
# const DEF_COLOR = ['#f1f1ef','#aaa','#444'];
 
# function color(k) {{ return COLORS[k] || DEF_COLOR; }}
# function parse(s) {{
#   const p = s.split('.');
#   return {{ type: p[2]||'?', id: p.slice(3).join('.')||s, full: s }};
# }}
# function byType(items) {{
#   const m = {{}};
#   items.forEach(i => {{ const p=parse(i); (m[p.type]=m[p.type]||[]).push(p); }});
#   return m;
# }}
 
# let activeKey = null;
 
# function renderKeys() {{
#   document.getElementById('keysGrid').innerHTML = ALL_KEYS.map(k => {{
#     const [bg, border, text] = color(k);
#     const cntA = (DICT_A[k]||[]).length;
#     const cntB = (DICT_B[k]||[]).length;
#     const isActive = activeKey === k;
#     return `<div class="key-card ${{isActive?'active':''}}"
#       style="background:${{bg}};border-color:${{border}};color:${{text}}"
#       onclick="selectKey('${{k}}')">
#       ${{k}}
#     </div>`;
#   }}).join('');
# }}
 
# function renderDetail() {{
#   const detail = document.getElementById('detail');
#   if (!activeKey) {{
#     detail.innerHTML = '<div class="detail-placeholder">Selecciona una clave para ver su detalle</div>';
#     return;
#   }}
#   const k = activeKey;
#   const [bg, border, text] = color(k);
 
#   function buildPanel(items, cls, label) {{
#     if (!items || !items.length)
#       return `<div class="dict-panel ${{cls}} empty-panel">Sin datos en este diccionario</div>`;
#     const grouped = byType(items);
#     const sections = Object.entries(grouped).map(([t, els]) => `
#       <div class="type-block">
#         <div class="type-lbl">${{t}} (${{els.length}})</div>
#         <div class="pills">${{els.map(e =>
#           `<span class="pill" title="${{e.full}}">${{e.id}}</span>`
#         ).join('')}}</div>
#       </div>`).join('');
#     return `<div class="dict-panel ${{cls}}"><h3>${{label}} — ${{items.length}} elementos</h3>${{sections}}</div>`;
#   }}
 
#   const totalA = (DICT_A[k]||[]).length;
#   const totalB = (DICT_B[k]||[]).length;
 
#   detail.innerHTML = `
#     <div class="detail-header">
#       <h2>${{k}}</h2>
#       <span class="detail-badge" style="background:${{bg}};border:1px solid ${{border}};color:${{text}}">${{LABEL_A}}: ${{totalA}}</span>
#       <span class="detail-badge" style="background:${{bg}};border:1px solid ${{border}};color:${{text}}">${{LABEL_B}}: ${{totalB}}</span>
#     </div>
#     <div class="dict-panels">
#       ${{buildPanel(DICT_A[k], 'a', LABEL_A)}}
#       ${{buildPanel(DICT_B[k], 'b', LABEL_B)}}
#     </div>`;
# }}
 
# function selectKey(k) {{
#   activeKey = activeKey === k ? null : k;
#   renderKeys();
#   renderDetail();
# }}
 
# renderKeys();
# </script>
# </body>
# </html>"""
#     return html

In [36]:
# def build_color_map(dict_a: dict, dict_b: dict) -> dict:
#     """
#     Genera un mapa de colores para cada clave según el número de errores.
    
#     Reglas:
#     - GRIS: Solo tiene errores en dict_a (dict_b está vacío)
#     - ROJO: +10 errores en dict_b
#     - AMARILLO: 6-10 errores en dict_b
#     - VERDE: 1-5 errores en dict_b
#     """
#     colors = {}
#     all_keys = set(dict_a.keys()) | set(dict_b.keys())
    
#     for key in all_keys:
#         errors_a = len(dict_a.get(key, []))
#         errors_b = len(dict_b.get(key, []))
        
#         # GRIS: Solo tiene errores en A (B está vacío)
#         if errors_b == 0:
#             colors[key] = ['#e0e0e0', '#9e9e9e', '#424242']  # Gris claro, borde gris medio, texto oscuro
        
#         # ROJO: +10 errores en B
#         elif errors_b > 10:
#             colors[key] = ['#b71c1c', '#ef5350', '#424242']  # Rojo claro, borde rojo, texto rojo oscuro
        
#         # AMARILLO: 6-10 errores en B
#         elif errors_b >= 6:
#             colors[key] = ['#f57f17', '#fbc02d', '#424242']  # Amarillo claro, borde amarillo, texto amarillo oscuro
        
#         # VERDE: 1-5 errores en B
#         else:  # errors_b >= 1 and errors_b <= 5
#             colors[key] = ['#2e7d32', '#66bb6a', '#424242']  # Verde claro, borde verde, texto verde oscuro
    
#     return colors

In [37]:
# def build_html(dict_a: dict, dict_b: dict,
#                label_a,
#                label_b,
#                output_title) -> str:
 
#     colors = build_color_map(dict_a, dict_b)
#     all_keys = sorted(set(dict_a) | set(dict_b))
 
#     colors_json   = json.dumps({k: list(v) for k, v in colors.items()}, ensure_ascii=False)
#     dict_a_json   = json.dumps(dict_a, ensure_ascii=False)
#     dict_b_json   = json.dumps(dict_b, ensure_ascii=False)
#     all_keys_json = json.dumps(all_keys, ensure_ascii=False)
 
#     total_a = sum(len(v) for v in dict_a.values())
#     total_b = sum(len(v) for v in dict_b.values())

 
#     html = f"""<!DOCTYPE html>
# <html lang="es">
# <head>
# <meta charset="UTF-8">
# <meta name="viewport" content="width=device-width, initial-scale=1.0">
# <title>{output_title}</title>
# <style>
# * {{ box-sizing: border-box; margin: 0; padding: 0; }}
# body {{ font-family: system-ui, sans-serif; background: #f4f4f2; color: #1a1a1a; }}
 
# header {{ background: #1a1a2e; color: #fff; padding: 1.25rem 2rem; }}
# header h1 {{ font-size: 1.3rem; font-weight: 500; }}
# header p  {{ font-size: 0.85rem; opacity: 0.55; margin-top: 3px; }}
 
# .stats {{ display: flex; gap: 12px; flex-wrap: wrap; padding: 1.25rem 2rem 0; }}
# .stat {{ background: #fff; border-radius: 8px; padding: .75rem 1.25rem;
#          border: 1px solid #e8e8e8; min-width: 120px; }}
# .stat .val {{ font-size: 1.6rem; font-weight: 600; }}
# .stat .lbl {{ font-size: 11px; color: #888; margin-top: 2px; }}
 
# .keys-section {{ padding: 1.25rem 2rem 0; }}
# .keys-section h2 {{ font-size: 11px; text-transform: uppercase; letter-spacing: .07em;
#                     color: #aaa; margin-bottom: 10px; }}
# .keys-grid {{ display: flex; flex-wrap: wrap; gap: 10px; }}
 
# .key-card {{ padding: 10px 22px; border-radius: 10px; border: 2px solid transparent;
#              cursor: pointer; font-size: 14px; font-weight: 700;
#              transition: transform .12s, box-shadow .12s; user-select: none; }}
# .key-card:hover {{ transform: translateY(-2px); box-shadow: 0 4px 14px rgba(0,0,0,.12); }}
# .key-card.active {{ box-shadow: 0 0 0 3px rgba(0,0,0,.18); }}
# .key-card .kc-counts {{ font-size: 11px; font-weight: 400; margin-top: 3px; opacity: .7; }}
 
# .detail {{ padding: 1.25rem 2rem 2rem; }}
# .detail-placeholder {{ display: flex; align-items: center; justify-content: center;
#                         height: 140px; color: #ccc; font-size: 14px; }}
 
# .detail-header {{ display: flex; align-items: center; gap: 10px; margin-bottom: 1.25rem;
#                   padding-bottom: 10px; border-bottom: 1px solid #e8e8e8; }}
# .detail-header h2 {{ font-size: 1.4rem; font-weight: 700; }}
# .detail-badge {{ font-size: 11px; padding: 3px 10px; border-radius: 99px; font-weight: 500; }}
 
# .dict-panels {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
# @media (max-width: 680px) {{ .dict-panels {{ grid-template-columns: 1fr; }} }}
 
# .dict-panel {{ background: #fff; border-radius: 10px; border: 1px solid #e8e8e8;
#                padding: 14px 16px; }}
# .dict-panel h3 {{ font-size: 12px; font-weight: 600; text-transform: uppercase;
#                   letter-spacing: .06em; margin-bottom: 10px; padding-bottom: 8px;
#                   border-bottom: 1px solid #f0f0f0; }}
# .dict-panel.a h3 {{ color: #2563eb; }}
# .dict-panel.b h3 {{ color: #059669; }}
# .dict-panel.empty-panel {{ color: #bbb; font-size: 13px; display: flex;
#                             align-items: center; justify-content: center; min-height: 80px; }}
 
# .type-block {{ margin-bottom: 10px; }}
# .type-lbl {{ font-size: 10px; text-transform: uppercase; letter-spacing: .06em;
#              color: #bbb; margin-bottom: 4px; }}
# .pills {{ display: flex; flex-wrap: wrap; gap: 5px; min-height: 30px;
#           padding: 4px; border-radius: 6px; transition: background 0.2s; }}
# .pills.drag-over {{ background: #e0e7ff; }}
# .pills.empty {{ border: 1px dashed #d0d0d0; min-height: 50px;
#                 display: flex; align-items: center; justify-content: center;
#                 color: #999; font-size: 11px; font-style: italic; }}
# .pill {{ font-size: 12px; padding: 3px 11px; border-radius: 99px;
#          background: #f5f5f3; border: 1px solid #e4e4e0; cursor: grab;
#          transition: all .15s; position: relative; }}
# .pill:hover {{ background: #e8e8e6; border-color: #d0d0cc; transform: translateY(-1px); }}
# .pill.has-comment {{ background: #fff4e6; border-color: #ffa500; }}
# .pill.active {{ background: #e0e7ff; border-color: #4f46e5; }}
# .pill.fixed {{ background: #d1fae5; border-color: #10b981; border-width: 2px; }}
# .pill.error {{ background: #fee2e2; border-color: #ef4444; border-width: 2px; }}
# .pill.dragging {{ opacity: 0.5; cursor: grabbing; }}

# .special-sections {{ display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 16px; }}
# @media (max-width: 680px) {{ .special-sections {{ grid-template-columns: 1fr; }} }}

# .fixed-section {{ padding: 12px; background: #f0fdf4;
#                   border: 2px dashed #10b981; border-radius: 8px; min-height: 80px; }}
# .fixed-section.drag-over {{ background: #dcfce7; border-style: solid; }}
# .fixed-section h4 {{ font-size: 11px; font-weight: 600; text-transform: uppercase;
#                      letter-spacing: .06em; color: #059669; margin-bottom: 8px; }}
# .fixed-section .pills-fixed {{ min-height: 40px; }}

# .error-section {{ padding: 12px; background: #fef2f2;
#                   border: 2px dashed #ef4444; border-radius: 8px; min-height: 80px; }}
# .error-section.drag-over {{ background: #fee2e2; border-style: solid; }}
# .error-section h4 {{ font-size: 11px; font-weight: 600; text-transform: uppercase;
#                      letter-spacing: .06em; color: #dc2626; margin-bottom: 8px; }}
# .error-section .pills-error {{ min-height: 40px; }}

# .section-empty {{ text-align: center; font-size: 12px; 
#                   padding: 12px; font-style: italic; }}
# .fixed-section .section-empty {{ color: #10b981; }}
# .error-section .section-empty {{ color: #ef4444; }}

# .normal-section {{ position: relative; }}
# .normal-section::after {{ content: ''; position: absolute; inset: -4px;
#                           border: 2px dashed transparent; border-radius: 8px;
#                           pointer-events: none; transition: border-color 0.2s; }}
# .normal-section.drag-over::after {{ border-color: #94a3b8; }}

# .comment-section {{ margin-top: 8px; padding: 12px; background: #fafafa;
#                      border-radius: 8px; border: 1px solid #e8e8e8;
#                      display: none; }}
# .comment-section.show {{ display: block; animation: slideDown 0.2s ease; }}
# @keyframes slideDown {{ from {{ opacity: 0; transform: translateY(-10px); }}
#                         to {{ opacity: 1; transform: translateY(0); }} }}

# .comment-header {{ display: flex; justify-content: space-between; align-items: center;
#                    margin-bottom: 8px; }}
# .comment-header .elem-name {{ font-size: 11px; font-weight: 600; color: #666; }}
# .comment-header .close-btn {{ background: none; border: none; color: #999;
#                               cursor: pointer; font-size: 18px; padding: 0;
#                               width: 24px; height: 24px; line-height: 24px;
#                               border-radius: 4px; transition: all .15s; }}
# .comment-header .close-btn:hover {{ background: #e8e8e8; color: #333; }}

# .comment-textarea {{ width: 100%; min-height: 80px; padding: 8px 10px;
#                      border: 1px solid #d0d0d0; border-radius: 6px;
#                      font-family: inherit; font-size: 13px; resize: vertical;
#                      transition: border-color .15s; }}
# .comment-textarea:focus {{ outline: none; border-color: #4f46e5; }}

# .comment-actions {{ display: flex; gap: 8px; margin-top: 8px; }}
# .comment-btn {{ padding: 6px 14px; border-radius: 6px; font-size: 12px;
#                 font-weight: 500; cursor: pointer; transition: all .15s;
#                 border: 1px solid; }}
# .comment-btn.save {{ background: #4f46e5; color: white; border-color: #4f46e5; }}
# .comment-btn.save:hover {{ background: #4338ca; }}
# .comment-btn.cancel {{ background: white; color: #666; border-color: #d0d0d0; }}
# .comment-btn.cancel:hover {{ background: #f5f5f5; }}
# .comment-btn.delete {{ background: #dc2626; color: white; border-color: #dc2626; }}
# .comment-btn.delete:hover {{ background: #b91c1c; }}

# .comment-display {{ font-size: 13px; color: #333; padding: 8px 10px;
#                     background: white; border-radius: 6px; border: 1px solid #e0e0e0;
#                     white-space: pre-wrap; word-wrap: break-word; }}
# </style>
# </head>
# <body>
 
# <header>
#   <h1>{output_title}</h1>
#   <p>Arrastra elementos entre secciones | Clic para comentarios | Gris = Solo errores de tecnólogo | Rojo =+ 10 errores en Topología MSE | NARANJA = 6-10 errores en Topología MSE | VERDE = 1-5 errores en Topología MSE</p>
# </header>
 
# <div class="stats">
#   <div class="stat"><div class="val">{len(all_keys)}</div><div class="lbl">Enclavamientos totales</div></div>
#   <div class="stat"><div class="val">{total_a}</div><div class="lbl">{label_a}</div></div>
#   <div class="stat"><div class="val">{total_b}</div><div class="lbl">{label_b}</div></div>
# </div>
 
# <div class="keys-section">
#   <h2>Claves — selecciona una</h2>
#   <div class="keys-grid" id="keysGrid"></div>
# </div>
 
# <div class="detail" id="detail">
#   <div class="detail-placeholder">Selecciona una clave para ver su detalle</div>
# </div>
 
# <script>
# const DICT_A    = {dict_a_json};
# const DICT_B    = {dict_b_json};
# const LABEL_A   = "{label_a}";
# const LABEL_B   = "{label_b}";
# const ALL_KEYS  = {all_keys_json};
# const COLORS    = {colors_json};
# const DEF_COLOR = ['#f1f1ef','#aaa','#444'];
 
# // Almacenar comentarios, elementos arreglados y errores en localStorage
# const COMMENTS_KEY = 'bcn_panel_comments';
# const FIXED_KEY = 'bcn_panel_fixed';
# const ERROR_KEY = 'bcn_panel_error';
# let comments = {{}};
# let fixedItems = {{}};
# let errorItems = {{}};

# function loadComments() {{
#   try {{
#     const stored = localStorage.getItem(COMMENTS_KEY);
#     if (stored) comments = JSON.parse(stored);
#   }} catch (e) {{ console.error('Error loading comments:', e); }}
# }}

# function saveComments() {{
#   try {{
#     localStorage.setItem(COMMENTS_KEY, JSON.stringify(comments));
#   }} catch (e) {{ console.error('Error saving comments:', e); }}
# }}

# function loadFixed() {{
#   try {{
#     const stored = localStorage.getItem(FIXED_KEY);
#     if (stored) fixedItems = JSON.parse(stored);
#   }} catch (e) {{ console.error('Error loading fixed items:', e); }}
# }}

# function saveFixed() {{
#   try {{
#     localStorage.setItem(FIXED_KEY, JSON.stringify(fixedItems));
#   }} catch (e) {{ console.error('Error saving fixed items:', e); }}
# }}

# function loadError() {{
#   try {{
#     const stored = localStorage.getItem(ERROR_KEY);
#     if (stored) errorItems = JSON.parse(stored);
#   }} catch (e) {{ console.error('Error loading error items:', e); }}
# }}

# function saveError() {{
#   try {{
#     localStorage.setItem(ERROR_KEY, JSON.stringify(errorItems));
#   }} catch (e) {{ console.error('Error saving error items:', e); }}
# }}

# function getCommentKey(key, elem) {{
#   return `${{key}}|${{elem}}`;
# }}

# function getFixedKey(key, panel) {{
#   return `${{key}}|${{panel}}`;
# }}

# function getErrorKey(key, panel) {{
#   return `${{key}}|${{panel}}|error`;
# }}

# function isFixed(key, panel, elem) {{
#   const fkey = getFixedKey(key, panel);
#   return fixedItems[fkey] && fixedItems[fkey].includes(elem);
# }}

# function isError(key, panel, elem) {{
#   const ekey = getErrorKey(key, panel);
#   return errorItems[ekey] && errorItems[ekey].includes(elem);
# }}

# function addFixed(key, panel, elem) {{
#   const fkey = getFixedKey(key, panel);
#   const ekey = getErrorKey(key, panel);
  
#   // Remover de errores si está ahí
#   if (errorItems[ekey]) {{
#     errorItems[ekey] = errorItems[ekey].filter(e => e !== elem);
#     if (errorItems[ekey].length === 0) delete errorItems[ekey];
#   }}
  
#   if (!fixedItems[fkey]) fixedItems[fkey] = [];
#   if (!fixedItems[fkey].includes(elem)) {{
#     fixedItems[fkey].push(elem);
#     saveFixed();
#     saveError();
#   }}
# }}

# function removeFixed(key, panel, elem) {{
#   const fkey = getFixedKey(key, panel);
#   if (fixedItems[fkey]) {{
#     fixedItems[fkey] = fixedItems[fkey].filter(e => e !== elem);
#     if (fixedItems[fkey].length === 0) delete fixedItems[fkey];
#     saveFixed();
#   }}
# }}

# function addError(key, panel, elem) {{
#   const ekey = getErrorKey(key, panel);
#   const fkey = getFixedKey(key, panel);
  
#   // Remover de arreglados si está ahí
#   if (fixedItems[fkey]) {{
#     fixedItems[fkey] = fixedItems[fkey].filter(e => e !== elem);
#     if (fixedItems[fkey].length === 0) delete fixedItems[fkey];
#   }}
  
#   if (!errorItems[ekey]) errorItems[ekey] = [];
#   if (!errorItems[ekey].includes(elem)) {{
#     errorItems[ekey].push(elem);
#     saveError();
#     saveFixed();
#   }}
# }}

# function removeError(key, panel, elem) {{
#   const ekey = getErrorKey(key, panel);
#   if (errorItems[ekey]) {{
#     errorItems[ekey] = errorItems[ekey].filter(e => e !== elem);
#     if (errorItems[ekey].length === 0) delete errorItems[ekey];
#     saveError();
#   }}
# }}

# loadComments();
# loadFixed();
# loadError();

# function color(k) {{ return COLORS[k] || DEF_COLOR; }}
# function parse(s) {{
#   const p = s.split('.');
#   return {{ type: p[2]||'?', id: p.slice(3).join('.')||s, full: s }};
# }}
# function byType(items) {{
#   const m = {{}};
#   items.forEach(i => {{ const p=parse(i); (m[p.type]=m[p.type]||[]).push(p); }});
#   return m;
# }}

# // Obtener todos los tipos únicos de un conjunto de items (incluso vacío)
# function getAllTypes(items) {{
#   const types = new Set();
#   items.forEach(item => {{
#     const p = parse(item);
#     types.add(p.type);
#   }});
#   return Array.from(types).sort();
# }}
 
# let activeKey = null;
# let activeElement = null;
 
# function renderKeys() {{
#   document.getElementById('keysGrid').innerHTML = ALL_KEYS.map(k => {{
#     const [bg, border, text] = color(k);
#     const cntA = (DICT_A[k]||[]).length;
#     const cntB = (DICT_B[k]||[]).length;
#     const isActive = activeKey === k;
#     return `<div class="key-card ${{isActive?'active':''}}"
#       style="background:${{bg}};border-color:${{border}};color:${{text}}"
#       onclick="selectKey('${{k}}')">
#       ${{k}}
#     </div>`;
#   }}).join('');
# }}
 
# function renderDetail() {{
#   const detail = document.getElementById('detail');
#   if (!activeKey) {{
#     detail.innerHTML = '<div class="detail-placeholder">Selecciona una clave para ver su detalle</div>';
#     return;
#   }}
#   const k = activeKey;
#   const [bg, border, text] = color(k);
 
#   function buildPanel(items, cls, label) {{
#     if (!items || !items.length)
#       return `<div class="dict-panel ${{cls}} empty-panel">Sin datos en este diccionario</div>`;
    
#     const fkey = getFixedKey(k, cls);
#     const ekey = getErrorKey(k, cls);
#     const fixed = fixedItems[fkey] || [];
#     const errors = errorItems[ekey] || [];
#     const notSpecial = items.filter(item => !fixed.includes(item) && !errors.includes(item));
    
#     // Obtener todos los tipos que existen en este panel (de todos los items originales)
#     const allTypes = getAllTypes(items);
    
#     const grouped = byType(notSpecial);
#     const groupedFixed = byType(fixed);
#     const groupedError = byType(errors);
    
#     // Crear secciones para TODOS los tipos, incluso si están vacíos
#     const sections = allTypes.map(t => {{
#       const els = grouped[t] || [];
#       const pillsHTML = els.length > 0 
#         ? els.map(e => {{
#             const commentKey = getCommentKey(k, e.full);
#             const hasComment = comments[commentKey];
#             const isActive = activeElement === e.full;
#             return `<span class="pill ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
#                          draggable="true"
#                          ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                          ondragend="handleDragEnd(event)"
#                          onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                          title="${{e.full}}">
#                       ${{e.id}}
#                     </span>`;
#           }}).join('')
#         : '';
      
#       const pillsClass = els.length === 0 ? 'pills empty' : 'pills';
#       const emptyText = els.length === 0 ? 'Arrastra aquí para devolver elementos' : '';
      
#       return `
#         <div class="type-block normal-section" 
#              ondragover="handleNormalDragOver(event)" 
#              ondragleave="handleNormalDragLeave(event)"
#              ondrop="handleNormalDrop(event, '${{cls}}')">
#           <div class="type-lbl">${{t}} (${{els.length}})</div>
#           <div class="${{pillsClass}}">${{pillsHTML || emptyText}}</div>
#           <div class="comment-container"></div>
#         </div>`;
#     }}).join('');
    
#     const fixedSections = Object.entries(groupedFixed).map(([t, els]) => `
#       <div class="type-block">
#         <div class="type-lbl">${{t}} (${{els.length}})</div>
#         <div class="pills">${{els.map(e => {{
#           const commentKey = getCommentKey(k, e.full);
#           const hasComment = comments[commentKey];
#           const isActive = activeElement === e.full;
#           return `<span class="pill fixed ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
#                        draggable="true"
#                        ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                        ondragend="handleDragEnd(event)"
#                        onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                        title="${{e.full}} (arrastra para mover)">
#                     ${{e.id}}
#                   </span>`;
#         }}).join('')}}</div>
#         <div class="comment-container"></div>
#       </div>`).join('');
    
#     const errorSections = Object.entries(groupedError).map(([t, els]) => `
#       <div class="type-block">
#         <div class="type-lbl">${{t}} (${{els.length}})</div>
#         <div class="pills">${{els.map(e => {{
#           const commentKey = getCommentKey(k, e.full);
#           const hasComment = comments[commentKey];
#           const isActive = activeElement === e.full;
#           return `<span class="pill error ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
#                        draggable="true"
#                        ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                        ondragend="handleDragEnd(event)"
#                        onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
#                        title="${{e.full}} (arrastra para mover)">
#                     ${{e.id}}
#                   </span>`;
#         }}).join('')}}</div>
#         <div class="comment-container"></div>
#       </div>`).join('');
    
#     const fixedContent = fixedSections || '<div class="section-empty">Arrastra elementos aquí para marcarlos como arreglados</div>';
#     const errorContent = errorSections || '<div class="section-empty">Arrastra elementos aquí para marcarlos como errores controlados</div>';
    
#     return `
#       <div class="dict-panel ${{cls}}">
#         <h3>${{label}} — ${{items.length}} elementos</h3>
#         ${{sections}}
#         <div class="special-sections">
#           <div class="fixed-section" 
#                ondragover="handleFixedDragOver(event)" 
#                ondragleave="handleFixedDragLeave(event)"
#                ondrop="handleFixedDrop(event, '${{cls}}')">
#             <h4>✓ Arreglados (${{fixed.length}})</h4>
#             <div class="pills-fixed">
#               ${{fixedContent}}
#             </div>
#           </div>
#           <div class="error-section" 
#                ondragover="handleErrorDragOver(event)" 
#                ondragleave="handleErrorDragLeave(event)"
#                ondrop="handleErrorDrop(event, '${{cls}}')">
#             <h4>⚠ Errores Controlados (${{errors.length}})</h4>
#             <div class="pills-error">
#               ${{errorContent}}
#             </div>
#           </div>
#         </div>
#       </div>`;
#   }}
 
#   const totalA = (DICT_A[k]||[]).length;
#   const totalB = (DICT_B[k]||[]).length;
 
#   detail.innerHTML = `
#     <div class="detail-header">
#       <h2>${{k}}</h2>
#       <span class="detail-badge" style="background:${{bg}};border:1px solid ${{border}};color:${{text}}">${{LABEL_A}}: ${{totalA}}</span>
#       <span class="detail-badge" style="background:${{bg}};border:1px solid ${{border}};color:${{text}}">${{LABEL_B}}: ${{totalB}}</span>
#     </div>
#     <div class="dict-panels">
#       ${{buildPanel(DICT_A[k], 'a', LABEL_A)}}
#       ${{buildPanel(DICT_B[k], 'b', LABEL_B)}}
#     </div>`;
  
#   // Si había un elemento activo, volver a mostrar su comentario
#   if (activeElement) {{
#     setTimeout(() => showCommentSection(activeKey, activeElement), 10);
#   }}
# }}

# let draggedElement = null;
# let draggedPanel = null;
# let draggedFromFixed = false;
# let draggedFromError = false;

# function handleDragStart(event, panel, elem) {{
#   draggedElement = elem;
#   draggedPanel = panel;
#   draggedFromFixed = isFixed(activeKey, panel, elem);
#   draggedFromError = isError(activeKey, panel, elem);
#   event.target.classList.add('dragging');
#   event.dataTransfer.effectAllowed = 'move';
# }}

# function handleDragEnd(event) {{
#   event.target.classList.remove('dragging');
# }}

# // Handlers para la sección normal
# function handleNormalDragOver(event) {{
#   if (!draggedFromFixed && !draggedFromError) return; // Solo permitir drop desde especiales
#   event.preventDefault();
#   event.dataTransfer.dropEffect = 'move';
#   event.currentTarget.classList.add('drag-over');
# }}

# function handleNormalDragLeave(event) {{
#   event.currentTarget.classList.remove('drag-over');
# }}

# function handleNormalDrop(event, targetPanel) {{
#   event.preventDefault();
#   event.currentTarget.classList.remove('drag-over');
  
#   if (draggedElement && draggedPanel === targetPanel) {{
#     if (draggedFromFixed) {{
#       removeFixed(activeKey, targetPanel, draggedElement);
#     }} else if (draggedFromError) {{
#       removeError(activeKey, targetPanel, draggedElement);
#     }}
#     renderDetail();
#   }}
  
#   draggedElement = null;
#   draggedPanel = null;
#   draggedFromFixed = false;
#   draggedFromError = false;
# }}

# // Handlers para la sección de arreglados
# function handleFixedDragOver(event) {{
#   if (draggedFromFixed) return; // No permitir drop desde arreglados a arreglados
#   event.preventDefault();
#   event.dataTransfer.dropEffect = 'move';
#   event.currentTarget.classList.add('drag-over');
# }}

# function handleFixedDragLeave(event) {{
#   event.currentTarget.classList.remove('drag-over');
# }}

# function handleFixedDrop(event, targetPanel) {{
#   event.preventDefault();
#   event.currentTarget.classList.remove('drag-over');
  
#   if (draggedElement && draggedPanel === targetPanel && !draggedFromFixed) {{
#     addFixed(activeKey, targetPanel, draggedElement);
#     renderDetail();
#   }}
  
#   draggedElement = null;
#   draggedPanel = null;
#   draggedFromFixed = false;
#   draggedFromError = false;
# }}

# // Handlers para la sección de errores
# function handleErrorDragOver(event) {{
#   if (draggedFromError) return; // No permitir drop desde errores a errores
#   event.preventDefault();
#   event.dataTransfer.dropEffect = 'move';
#   event.currentTarget.classList.add('drag-over');
# }}

# function handleErrorDragLeave(event) {{
#   event.currentTarget.classList.remove('drag-over');
# }}

# function handleErrorDrop(event, targetPanel) {{
#   event.preventDefault();
#   event.currentTarget.classList.remove('drag-over');
  
#   if (draggedElement && draggedPanel === targetPanel && !draggedFromError) {{
#     addError(activeKey, targetPanel, draggedElement);
#     renderDetail();
#   }}
  
#   draggedElement = null;
#   draggedPanel = null;
#   draggedFromFixed = false;
#   draggedFromError = false;
# }}

# function toggleComment(key, elem) {{
#   event.stopPropagation();
  
#   if (activeElement === elem) {{
#     closeComment();
#   }} else {{
#     activeElement = elem;
#     renderDetail();
#   }}
# }}

# function showCommentSection(key, elem) {{
#   const commentKey = getCommentKey(key, elem);
#   const existingComment = comments[commentKey] || '';
  
#   // Encontrar el pill correspondiente
#   const pills = document.querySelectorAll('.pill');
#   let targetPill = null;
#   pills.forEach(pill => {{
#     if (pill.title === elem || pill.title.startsWith(elem + ' ')) {{
#       targetPill = pill;
#     }}
#   }});
  
#   if (!targetPill) return;
  
#   // Crear la sección de comentarios
#   const typeBlock = targetPill.closest('.type-block');
#   let commentContainer = typeBlock.querySelector('.comment-container');
  
#   if (!commentContainer) {{
#     commentContainer = document.createElement('div');
#     commentContainer.className = 'comment-container';
#     typeBlock.appendChild(commentContainer);
#   }}
  
#   const isEditing = !existingComment || existingComment === '';
#   const btnText = existingComment ? 'Actualizar' : 'Guardar';
#   const deleteBtn = existingComment ? `<button class="comment-btn delete" onclick="deleteComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')" >Eliminar</button>` : '';
  
#   const editingHTML = `
#     <textarea class="comment-textarea" id="commentText" placeholder="Escribe tu comentario aquí...">${{existingComment}}</textarea>
#     <div class="comment-actions">
#       <button class="comment-btn save" onclick="saveComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')">${{btnText}}</button>
#       <button class="comment-btn cancel" onclick="closeComment()">Cancelar</button>
#       ${{deleteBtn}}
#     </div>
#   `;
  
#   const displayHTML = `
#     <div class="comment-display">${{existingComment}}</div>
#     <div class="comment-actions">
#       <button class="comment-btn save" onclick="editComment()">Editar</button>
#       <button class="comment-btn delete" onclick="deleteComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')" >Eliminar</button>
#     </div>
#   `;
  
#   commentContainer.innerHTML = `
#     <div class="comment-section show">
#       <div class="comment-header">
#         <span class="elem-name">${{elem}}</span>
#         <button class="close-btn" onclick="closeComment()">×</button>
#       </div>
#       ${{isEditing ? editingHTML : displayHTML}}
#     </div>
#   `;
  
#   if (isEditing) {{
#     setTimeout(() => {{
#       const textarea = document.getElementById('commentText');
#       if (textarea) textarea.focus();
#     }}, 50);
#   }}
# }}

# function editComment() {{
#   if (activeKey && activeElement) {{
#     renderDetail();
#   }}
# }}

# function saveComment(key, elem) {{
#   const textarea = document.getElementById('commentText');
#   if (!textarea) return;
  
#   const comment = textarea.value.trim();
#   const commentKey = getCommentKey(key, elem);
  
#   if (comment) {{
#     comments[commentKey] = comment;
#     saveComments();
#   }} else {{
#     delete comments[commentKey];
#     saveComments();
#   }}
  
#   renderDetail();
# }}

# function deleteComment(key, elem) {{
#   if (!confirm('¿Estás seguro de que quieres eliminar este comentario?')) return;
  
#   const commentKey = getCommentKey(key, elem);
#   delete comments[commentKey];
#   saveComments();
#   activeElement = null;
#   renderDetail();
# }}

# function closeComment() {{
#   activeElement = null;
#   renderDetail();
# }}
 
# function selectKey(k) {{
#   activeKey = activeKey === k ? null : k;
#   activeElement = null;
#   renderKeys();
#   renderDetail();
# }}
 
# renderKeys();
# </script>
# </body>
# </html>"""
#     return html

In [38]:
import json


def build_color_map(dict_a: dict, dict_b: dict) -> dict:
    colors = {}
    all_keys = set(dict_a.keys()) | set(dict_b.keys())
    
    for key in all_keys:
        errors_b = len(dict_b.get(key, []))
        
        if errors_b == 0:
            colors[key] = ['#e0e0e0', '#9e9e9e', '#424242']
        elif errors_b > 10:
            colors[key] = ['#b71c1c', '#ef5350', '#424242']
        elif errors_b >= 6:
            colors[key] = ['#f57f17', '#fbc02d', '#424242']
        else:
            colors[key] = ['#2e7d32', '#66bb6a', '#424242']
    
    return colors


def build_html(dict_a: dict, dict_b: dict,
               label_a,
               label_b,
               output_title,
               initial_comments=None,
               initial_fixed=None,
               initial_errors=None) -> str:
 
    colors = build_color_map(dict_a, dict_b)
    all_keys = sorted(set(dict_a) | set(dict_b))
 
    colors_json   = json.dumps({k: list(v) for k, v in colors.items()}, ensure_ascii=False)
    dict_a_json   = json.dumps(dict_a, ensure_ascii=False)
    dict_b_json   = json.dumps(dict_b, ensure_ascii=False)
    all_keys_json = json.dumps(all_keys, ensure_ascii=False)

    # Estado inicial embebido (para compartir el HTML con cambios ya hechos)
    initial_comments_json = json.dumps(initial_comments or {}, ensure_ascii=False)
    initial_fixed_json    = json.dumps(initial_fixed or {}, ensure_ascii=False)
    initial_errors_json   = json.dumps(initial_errors or {}, ensure_ascii=False)
 
    total_a = sum(len(v) for v in dict_a.values())
    total_b = sum(len(v) for v in dict_b.values())

    html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{output_title}</title>
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: system-ui, sans-serif; background: #f4f4f2; color: #1a1a1a; }}
 
header {{ background: #1a1a2e; color: #fff; padding: 1.25rem 2rem;
          display: flex; align-items: flex-start; justify-content: space-between; gap: 1rem; }}
header .header-left h1 {{ font-size: 1.3rem; font-weight: 500; }}
header .header-left p  {{ font-size: 0.85rem; opacity: 0.55; margin-top: 3px; }}

.export-btn {{ flex-shrink: 0; background: #4f46e5; color: white; border: none;
               padding: 8px 18px; border-radius: 8px; font-size: 13px; font-weight: 600;
               cursor: pointer; transition: background .15s; white-space: nowrap;
               align-self: center; }}
.export-btn:hover {{ background: #4338ca; }}
.export-btn:active {{ background: #3730a3; }}
 
.stats {{ display: flex; gap: 12px; flex-wrap: wrap; padding: 1.25rem 2rem 0; }}
.stat {{ background: #fff; border-radius: 8px; padding: .75rem 1.25rem;
         border: 1px solid #e8e8e8; min-width: 120px; }}
.stat .val {{ font-size: 1.6rem; font-weight: 600; }}
.stat .lbl {{ font-size: 11px; color: #888; margin-top: 2px; font-weight: 700; }}
 
.keys-section {{ padding: 1.25rem 2rem 0; }}
.keys-section h2 {{ font-size: 11px; text-transform: uppercase; letter-spacing: .07em;
                    color: #aaa; margin-bottom: 10px; }}
.keys-grid {{ display: flex; flex-wrap: wrap; gap: 10px; }}
 
.key-card {{ padding: 10px 22px; border-radius: 10px; border: 2px solid transparent;
             cursor: pointer; font-size: 14px; font-weight: 700;
             transition: transform .12s, box-shadow .12s; user-select: none; }}
.key-card:hover {{ transform: translateY(-2px); box-shadow: 0 4px 14px rgba(0,0,0,.12); }}
.key-card.active {{ box-shadow: 0 0 0 3px rgba(0,0,0,.18); }}
.key-card .kc-counts {{ font-size: 11px; font-weight: 400; margin-top: 3px; opacity: .7; }}
 
.detail {{ padding: 1.25rem 2rem 2rem; }}
.detail-placeholder {{ display: flex; align-items: center; justify-content: center;
                        height: 140px; color: #ccc; font-size: 14px; }}
 
.detail-header {{ display: flex; align-items: center; gap: 10px; margin-bottom: 1.25rem;
                  padding-bottom: 10px; border-bottom: 1px solid #e8e8e8; }}
.detail-header h2 {{ font-size: 1.4rem; font-weight: 700; }}
.detail-badge {{ font-size: 11px; padding: 3px 10px; border-radius: 99px; font-weight: 700; }}
 
.dict-panels {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
@media (max-width: 680px) {{ .dict-panels {{ grid-template-columns: 1fr; }} }}
 
.dict-panel {{ background: #fff; border-radius: 10px; border: 1px solid #e8e8e8;
               padding: 14px 16px; }}
.dict-panel h3 {{ font-size: 12px; font-weight: 700; text-transform: uppercase;
                  letter-spacing: .06em; margin-bottom: 10px; padding-bottom: 8px;
                  border-bottom: 1px solid #f0f0f0; }}
.dict-panel.a h3 {{ color: #2563eb; }}
.dict-panel.b h3 {{ color: #059669; }}
.dict-panel.empty-panel {{ color: #bbb; font-size: 13px; display: flex;
                            align-items: center; justify-content: center; min-height: 80px; }}
 
.type-block {{ margin-bottom: 10px; }}
.type-lbl {{ font-size: 10px; text-transform: uppercase; letter-spacing: .06em;
             color: #bbb; margin-bottom: 4px; }}
.pills {{ display: flex; flex-wrap: wrap; gap: 5px; min-height: 30px;
          padding: 4px; border-radius: 6px; transition: background 0.2s; }}
.pills.drag-over {{ background: #e0e7ff; }}
.pills.empty {{ border: 1px dashed #d0d0d0; min-height: 50px;
                display: flex; align-items: center; justify-content: center;
                color: #999; font-size: 11px; font-style: italic; }}
.pill {{ font-size: 12px; padding: 3px 11px; border-radius: 99px;
         background: #f5f5f3; border: 1px solid #e4e4e0; cursor: grab;
         transition: all .15s; position: relative; }}
.pill:hover {{ background: #e8e8e6; border-color: #d0d0cc; transform: translateY(-1px); }}
.pill.has-comment {{ background: #fff4e6; border-color: #ffa500; }}
.pill.active {{ background: #e0e7ff; border-color: #4f46e5; }}
.pill.fixed {{ background: #d1fae5; border-color: #10b981; border-width: 2px; }}
.pill.error {{ background: #fee2e2; border-color: #ef4444; border-width: 2px; }}
.pill.dragging {{ opacity: 0.5; cursor: grabbing; }}

.special-sections {{ display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 16px; }}
@media (max-width: 680px) {{ .special-sections {{ grid-template-columns: 1fr; }} }}

.fixed-section {{ padding: 12px; background: #f0fdf4;
                  border: 2px dashed #10b981; border-radius: 8px; min-height: 80px; }}
.fixed-section.drag-over {{ background: #dcfce7; border-style: solid; }}
.fixed-section h4 {{ font-size: 11px; font-weight: 600; text-transform: uppercase;
                     letter-spacing: .06em; color: #059669; margin-bottom: 8px; }}
.fixed-section .pills-fixed {{ min-height: 40px; }}

.error-section {{ padding: 12px; background: #fef2f2;
                  border: 2px dashed #ef4444; border-radius: 8px; min-height: 80px; }}
.error-section.drag-over {{ background: #fee2e2; border-style: solid; }}
.error-section h4 {{ font-size: 11px; font-weight: 600; text-transform: uppercase;
                     letter-spacing: .06em; color: #dc2626; margin-bottom: 8px; }}
.error-section .pills-error {{ min-height: 40px; }}

.section-empty {{ text-align: center; font-size: 12px; 
                  padding: 12px; font-style: italic; }}
.fixed-section .section-empty {{ color: #10b981; }}
.error-section .section-empty {{ color: #ef4444; }}

.normal-section {{ position: relative; }}
.normal-section::after {{ content: ''; position: absolute; inset: -4px;
                          border: 2px dashed transparent; border-radius: 8px;
                          pointer-events: none; transition: border-color 0.2s; }}
.normal-section.drag-over::after {{ border-color: #94a3b8; }}

.comment-section {{ margin-top: 8px; padding: 12px; background: #fafafa;
                     border-radius: 8px; border: 1px solid #e8e8e8;
                     display: none; }}
.comment-section.show {{ display: block; animation: slideDown 0.2s ease; }}
@keyframes slideDown {{ from {{ opacity: 0; transform: translateY(-10px); }}
                        to {{ opacity: 1; transform: translateY(0); }} }}

.comment-header {{ display: flex; justify-content: space-between; align-items: center;
                   margin-bottom: 8px; }}
.comment-header .elem-name {{ font-size: 11px; font-weight: 600; color: #666; }}
.comment-header .close-btn {{ background: none; border: none; color: #999;
                              cursor: pointer; font-size: 18px; padding: 0;
                              width: 24px; height: 24px; line-height: 24px;
                              border-radius: 4px; transition: all .15s; }}
.comment-header .close-btn:hover {{ background: #e8e8e8; color: #333; }}

.comment-textarea {{ width: 100%; min-height: 80px; padding: 8px 10px;
                     border: 1px solid #d0d0d0; border-radius: 6px;
                     font-family: inherit; font-size: 13px; resize: vertical;
                     transition: border-color .15s; }}
.comment-textarea:focus {{ outline: none; border-color: #4f46e5; }}

.comment-actions {{ display: flex; gap: 8px; margin-top: 8px; }}
.comment-btn {{ padding: 6px 14px; border-radius: 6px; font-size: 12px;
                font-weight: 500; cursor: pointer; transition: all .15s;
                border: 1px solid; }}
.comment-btn.save {{ background: #4f46e5; color: white; border-color: #4f46e5; }}
.comment-btn.save:hover {{ background: #4338ca; }}
.comment-btn.cancel {{ background: white; color: #666; border-color: #d0d0d0; }}
.comment-btn.cancel:hover {{ background: #f5f5f5; }}
.comment-btn.delete {{ background: #dc2626; color: white; border-color: #dc2626; }}
.comment-btn.delete:hover {{ background: #b91c1c; }}

.comment-display {{ font-size: 13px; color: #333; padding: 8px 10px;
                    background: white; border-radius: 6px; border: 1px solid #e0e0e0;
                    white-space: pre-wrap; word-wrap: break-word; }}

/* Toast de confirmación */
.toast {{ position: fixed; bottom: 24px; right: 24px; background: #1a1a2e; color: white;
          padding: 12px 20px; border-radius: 8px; font-size: 13px; font-weight: 500;
          opacity: 0; transform: translateY(10px); transition: all .3s;
          pointer-events: none; z-index: 9999; }}
.toast.show {{ opacity: 1; transform: translateY(0); }}
</style>
</head>
<body>

<div class="toast" id="toast">✓ HTML exportado correctamente</div>
 
<header>
  <div class="header-left">
    <h1>{output_title}</h1>
    <p>Arrastra elementos entre secciones | Clic para comentarios | Gris = Solo errores de tecnólogo | Rojo = +10 errores en Topología MSE | NARANJA = 6-10 errores en Topología MSE | VERDE = 1-5 errores en Topología MSE</p>
  </div>
  <button class="export-btn" onclick="exportHTML()">⬇ Exportar HTML</button>
</header>
 
<div class="stats">
  <div class="stat"><div class="val">{len(all_keys)}</div><div class="lbl">Enclavamientos totales</div></div>
  <div class="stat"><div class="val">{total_a}</div><div class="lbl">{label_a}</div></div>
  <div class="stat"><div class="val">{total_b}</div><div class="lbl">{label_b}</div></div>
</div>
 
<div class="keys-section">
  <h2>Claves — selecciona una</h2>
  <div class="keys-grid" id="keysGrid"></div>
</div>
 
<div class="detail" id="detail">
  <div class="detail-placeholder">Selecciona una clave para ver su detalle</div>
</div>
 
<script>
const DICT_A    = {dict_a_json};
const DICT_B    = {dict_b_json};
const LABEL_A   = "{label_a}";
const LABEL_B   = "{label_b}";
const ALL_KEYS  = {all_keys_json};
const COLORS    = {colors_json};
const DEF_COLOR = ['#f1f1ef','#aaa','#444'];

// ── Estado embebido al exportar ──────────────────────────────────────────────
// Estos valores se sobreescriben cada vez que se exporta el HTML.
// Si están vacíos ({{}}) se usa localStorage como fallback normal.
const EMBEDDED_COMMENTS  = {initial_comments_json};
const EMBEDDED_FIXED     = {initial_fixed_json};
const EMBEDDED_ERRORS    = {initial_errors_json};
// ─────────────────────────────────────────────────────────────────────────────
 
const COMMENTS_KEY = 'bcn_panel_comments';
const FIXED_KEY    = 'bcn_panel_fixed';
const ERROR_KEY    = 'bcn_panel_error';

let comments   = {{}};
let fixedItems  = {{}};
let errorItems  = {{}};

function loadComments() {{
  // Prioridad: datos embebidos > localStorage
  if (Object.keys(EMBEDDED_COMMENTS).length > 0) {{
    comments = JSON.parse(JSON.stringify(EMBEDDED_COMMENTS));
    // Sincronizar también localStorage para que ediciones futuras persistan
    try {{ localStorage.setItem(COMMENTS_KEY, JSON.stringify(comments)); }} catch(e) {{}}
    return;
  }}
  try {{
    const stored = localStorage.getItem(COMMENTS_KEY);
    if (stored) comments = JSON.parse(stored);
  }} catch (e) {{ console.error('Error loading comments:', e); }}
}}

function saveComments() {{
  try {{ localStorage.setItem(COMMENTS_KEY, JSON.stringify(comments)); }} catch (e) {{}}
}}

function loadFixed() {{
  if (Object.keys(EMBEDDED_FIXED).length > 0) {{
    fixedItems = JSON.parse(JSON.stringify(EMBEDDED_FIXED));
    try {{ localStorage.setItem(FIXED_KEY, JSON.stringify(fixedItems)); }} catch(e) {{}}
    return;
  }}
  try {{
    const stored = localStorage.getItem(FIXED_KEY);
    if (stored) fixedItems = JSON.parse(stored);
  }} catch (e) {{ console.error('Error loading fixed items:', e); }}
}}

function saveFixed() {{
  try {{ localStorage.setItem(FIXED_KEY, JSON.stringify(fixedItems)); }} catch (e) {{}}
}}

function loadError() {{
  if (Object.keys(EMBEDDED_ERRORS).length > 0) {{
    errorItems = JSON.parse(JSON.stringify(EMBEDDED_ERRORS));
    try {{ localStorage.setItem(ERROR_KEY, JSON.stringify(errorItems)); }} catch(e) {{}}
    return;
  }}
  try {{
    const stored = localStorage.getItem(ERROR_KEY);
    if (stored) errorItems = JSON.parse(stored);
  }} catch (e) {{ console.error('Error loading error items:', e); }}
}}

function saveError() {{
  try {{ localStorage.setItem(ERROR_KEY, JSON.stringify(errorItems)); }} catch (e) {{}}
}}

// ── Exportar HTML con estado embebido ────────────────────────────────────────
function exportHTML() {{
  // Serializar el estado actual
  const commentsJSON = JSON.stringify(comments);
  const fixedJSON    = JSON.stringify(fixedItems);
  const errorsJSON   = JSON.stringify(errorItems);

  // Leer el HTML actual del documento
  let src = document.documentElement.outerHTML;

  // Reemplazar los bloques EMBEDDED_* con los valores actuales.
  // Usamos los marcadores de línea completa para evitar colisiones.
  src = src.replace(
    /const EMBEDDED_COMMENTS\s*=\s*\{{[^;]*\}};/,
    `const EMBEDDED_COMMENTS  = ${{commentsJSON}};`
  );
  src = src.replace(
    /const EMBEDDED_FIXED\s*=\s*\{{[^;]*\}};/,
    `const EMBEDDED_FIXED     = ${{fixedJSON}};`
  );
  src = src.replace(
    /const EMBEDDED_ERRORS\s*=\s*\{{[^;]*\}};/,
    `const EMBEDDED_ERRORS    = ${{errorsJSON}};`
  );

  // Crear y descargar el fichero
  const blob = new Blob([src], {{ type: 'text/html;charset=utf-8' }});
  const url  = URL.createObjectURL(blob);
  const a    = document.createElement('a');
  a.href     = url;
  a.download = '{output_title}.html';
  document.body.appendChild(a);
  a.click();
  document.body.removeChild(a);
  URL.revokeObjectURL(url);

  // Mostrar toast
  const toast = document.getElementById('toast');
  toast.classList.add('show');
  setTimeout(() => toast.classList.remove('show'), 2500);
}}
// ─────────────────────────────────────────────────────────────────────────────

function getCommentKey(key, elem) {{ return `${{key}}|${{elem}}`; }}
function getFixedKey(key, panel)  {{ return `${{key}}|${{panel}}`; }}
function getErrorKey(key, panel)  {{ return `${{key}}|${{panel}}|error`; }}

function isFixed(key, panel, elem) {{
  const fkey = getFixedKey(key, panel);
  return fixedItems[fkey] && fixedItems[fkey].includes(elem);
}}

function isError(key, panel, elem) {{
  const ekey = getErrorKey(key, panel);
  return errorItems[ekey] && errorItems[ekey].includes(elem);
}}

function addFixed(key, panel, elem) {{
  const fkey = getFixedKey(key, panel);
  const ekey = getErrorKey(key, panel);
  if (errorItems[ekey]) {{
    errorItems[ekey] = errorItems[ekey].filter(e => e !== elem);
    if (errorItems[ekey].length === 0) delete errorItems[ekey];
  }}
  if (!fixedItems[fkey]) fixedItems[fkey] = [];
  if (!fixedItems[fkey].includes(elem)) {{
    fixedItems[fkey].push(elem);
    saveFixed(); saveError();
  }}
}}

function removeFixed(key, panel, elem) {{
  const fkey = getFixedKey(key, panel);
  if (fixedItems[fkey]) {{
    fixedItems[fkey] = fixedItems[fkey].filter(e => e !== elem);
    if (fixedItems[fkey].length === 0) delete fixedItems[fkey];
    saveFixed();
  }}
}}

function addError(key, panel, elem) {{
  const ekey = getErrorKey(key, panel);
  const fkey = getFixedKey(key, panel);
  if (fixedItems[fkey]) {{
    fixedItems[fkey] = fixedItems[fkey].filter(e => e !== elem);
    if (fixedItems[fkey].length === 0) delete fixedItems[fkey];
  }}
  if (!errorItems[ekey]) errorItems[ekey] = [];
  if (!errorItems[ekey].includes(elem)) {{
    errorItems[ekey].push(elem);
    saveError(); saveFixed();
  }}
}}

function removeError(key, panel, elem) {{
  const ekey = getErrorKey(key, panel);
  if (errorItems[ekey]) {{
    errorItems[ekey] = errorItems[ekey].filter(e => e !== elem);
    if (errorItems[ekey].length === 0) delete errorItems[ekey];
    saveError();
  }}
}}

loadComments();
loadFixed();
loadError();

function color(k) {{ return COLORS[k] || DEF_COLOR; }}
function parse(s) {{
  const p = s.split('.');
  return {{ type: p[2]||'?', id: p.slice(3).join('.')||s, full: s }};
}}
function byType(items) {{
  const m = {{}};
  items.forEach(i => {{ const p=parse(i); (m[p.type]=m[p.type]||[]).push(p); }});
  return m;
}}
function getAllTypes(items) {{
  const types = new Set();
  items.forEach(item => {{ const p = parse(item); types.add(p.type); }});
  return Array.from(types).sort();
}}
 
let activeKey = null;
let activeElement = null;
 
function renderKeys() {{
  document.getElementById('keysGrid').innerHTML = ALL_KEYS.map(k => {{
    const [bg, border, text] = color(k);
    const isActive = activeKey === k;
    return `<div class="key-card ${{isActive?'active':''}}"
      style="background:${{bg}};border-color:${{border}};color:${{text}}"
      onclick="selectKey('${{k}}')">
      ${{k}}
    </div>`;
  }}).join('');
}}
 
function renderDetail() {{
  const detail = document.getElementById('detail');
  if (!activeKey) {{
    detail.innerHTML = '<div class="detail-placeholder">Selecciona una clave para ver su detalle</div>';
    return;
  }}
  const k = activeKey;
  const [bg, border, text] = color(k);
 
  function buildPanel(items, cls, label) {{
    if (!items || !items.length)
      return `<div class="dict-panel ${{cls}} empty-panel">Sin datos en este diccionario</div>`;
    
    const fkey = getFixedKey(k, cls);
    const ekey = getErrorKey(k, cls);
    const fixed  = fixedItems[fkey] || [];
    const errors = errorItems[ekey] || [];
    const notSpecial = items.filter(item => !fixed.includes(item) && !errors.includes(item));
    const allTypes = getAllTypes(items);
    const grouped      = byType(notSpecial);
    const groupedFixed = byType(fixed);
    const groupedError = byType(errors);
    
    const sections = allTypes.map(t => {{
      const els = grouped[t] || [];
      const pillsHTML = els.length > 0 
        ? els.map(e => {{
            const commentKey = getCommentKey(k, e.full);
            const hasComment = comments[commentKey];
            const isActive = activeElement === e.full;
            return `<span class="pill ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
                         draggable="true"
                         ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                         ondragend="handleDragEnd(event)"
                         onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                         title="${{e.full}}">
                      ${{e.id}}
                    </span>`;
          }}).join('')
        : '';
      const pillsClass = els.length === 0 ? 'pills empty' : 'pills';
      const emptyText  = els.length === 0 ? 'Arrastra aquí para devolver elementos' : '';
      return `
        <div class="type-block normal-section" 
             ondragover="handleNormalDragOver(event)" 
             ondragleave="handleNormalDragLeave(event)"
             ondrop="handleNormalDrop(event, '${{cls}}')">
          <div class="type-lbl">${{t}} (${{els.length}})</div>
          <div class="${{pillsClass}}">${{pillsHTML || emptyText}}</div>
          <div class="comment-container"></div>
        </div>`;
    }}).join('');
    
    const fixedSections = Object.entries(groupedFixed).map(([t, els]) => `
      <div class="type-block">
        <div class="type-lbl">${{t}} (${{els.length}})</div>
        <div class="pills">${{els.map(e => {{
          const commentKey = getCommentKey(k, e.full);
          const hasComment = comments[commentKey];
          const isActive = activeElement === e.full;
          return `<span class="pill fixed ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
                       draggable="true"
                       ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                       ondragend="handleDragEnd(event)"
                       onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                       title="${{e.full}} (arrastra para mover)">
                    ${{e.id}}
                  </span>`;
        }}).join('')}}</div>
        <div class="comment-container"></div>
      </div>`).join('');
    
    const errorSections = Object.entries(groupedError).map(([t, els]) => `
      <div class="type-block">
        <div class="type-lbl">${{t}} (${{els.length}})</div>
        <div class="pills">${{els.map(e => {{
          const commentKey = getCommentKey(k, e.full);
          const hasComment = comments[commentKey];
          const isActive = activeElement === e.full;
          return `<span class="pill error ${{hasComment?'has-comment':''}} ${{isActive?'active':''}}" 
                       draggable="true"
                       ondragstart="handleDragStart(event, '${{cls}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                       ondragend="handleDragEnd(event)"
                       onclick="toggleComment('${{k}}', '${{e.full.replace(/'/g, "\\\\'")}}')"
                       title="${{e.full}} (arrastra para mover)">
                    ${{e.id}}
                  </span>`;
        }}).join('')}}</div>
        <div class="comment-container"></div>
      </div>`).join('');
    
    const fixedContent = fixedSections || '<div class="section-empty">Arrastra elementos aquí para marcarlos como arreglados</div>';
    const errorContent = errorSections || '<div class="section-empty">Arrastra elementos aquí para marcarlos como errores controlados</div>';
    
    return `
      <div class="dict-panel ${{cls}}">
        <h3>${{label}} — ${{items.length}} elementos</h3>
        ${{sections}}
        <div class="special-sections">
          <div class="fixed-section" 
               ondragover="handleFixedDragOver(event)" 
               ondragleave="handleFixedDragLeave(event)"
               ondrop="handleFixedDrop(event, '${{cls}}')">
            <h4>✓ Arreglados (${{fixed.length}})</h4>
            <div class="pills-fixed">${{fixedContent}}</div>
          </div>
          <div class="error-section" 
               ondragover="handleErrorDragOver(event)" 
               ondragleave="handleErrorDragLeave(event)"
               ondrop="handleErrorDrop(event, '${{cls}}')">
            <h4>⚠ Errores Controlados (${{errors.length}})</h4>
            <div class="pills-error">${{errorContent}}</div>
          </div>
        </div>
      </div>`;
  }}
 
  const totalA = (DICT_A[k]||[]).length;
  const totalB = (DICT_B[k]||[]).length;
 
  detail.innerHTML = `
    <div class="detail-header">
      <h2>${{k}}</h2>
    </div>
    <div class="dict-panels">
      ${{buildPanel(DICT_A[k], 'a', LABEL_A)}}
      ${{buildPanel(DICT_B[k], 'b', LABEL_B)}}
    </div>`;
  
  if (activeElement) {{
    setTimeout(() => showCommentSection(activeKey, activeElement), 10);
  }}
}}

let draggedElement = null;
let draggedPanel   = null;
let draggedFromFixed = false;
let draggedFromError = false;

function handleDragStart(event, panel, elem) {{
  draggedElement   = elem;
  draggedPanel     = panel;
  draggedFromFixed = isFixed(activeKey, panel, elem);
  draggedFromError = isError(activeKey, panel, elem);
  event.target.classList.add('dragging');
  event.dataTransfer.effectAllowed = 'move';
}}

function handleDragEnd(event) {{ event.target.classList.remove('dragging'); }}

function handleNormalDragOver(event) {{
  if (!draggedFromFixed && !draggedFromError) return;
  event.preventDefault();
  event.dataTransfer.dropEffect = 'move';
  event.currentTarget.classList.add('drag-over');
}}
function handleNormalDragLeave(event) {{ event.currentTarget.classList.remove('drag-over'); }}
function handleNormalDrop(event, targetPanel) {{
  event.preventDefault();
  event.currentTarget.classList.remove('drag-over');
  if (draggedElement && draggedPanel === targetPanel) {{
    if (draggedFromFixed)      removeFixed(activeKey, targetPanel, draggedElement);
    else if (draggedFromError) removeError(activeKey, targetPanel, draggedElement);
    renderDetail();
  }}
  draggedElement = null; draggedPanel = null;
  draggedFromFixed = false; draggedFromError = false;
}}

function handleFixedDragOver(event) {{
  if (draggedFromFixed) return;
  event.preventDefault();
  event.dataTransfer.dropEffect = 'move';
  event.currentTarget.classList.add('drag-over');
}}
function handleFixedDragLeave(event) {{ event.currentTarget.classList.remove('drag-over'); }}
function handleFixedDrop(event, targetPanel) {{
  event.preventDefault();
  event.currentTarget.classList.remove('drag-over');
  if (draggedElement && draggedPanel === targetPanel && !draggedFromFixed) {{
    addFixed(activeKey, targetPanel, draggedElement);
    renderDetail();
  }}
  draggedElement = null; draggedPanel = null;
  draggedFromFixed = false; draggedFromError = false;
}}

function handleErrorDragOver(event) {{
  if (draggedFromError) return;
  event.preventDefault();
  event.dataTransfer.dropEffect = 'move';
  event.currentTarget.classList.add('drag-over');
}}
function handleErrorDragLeave(event) {{ event.currentTarget.classList.remove('drag-over'); }}
function handleErrorDrop(event, targetPanel) {{
  event.preventDefault();
  event.currentTarget.classList.remove('drag-over');
  if (draggedElement && draggedPanel === targetPanel && !draggedFromError) {{
    addError(activeKey, targetPanel, draggedElement);
    renderDetail();
  }}
  draggedElement = null; draggedPanel = null;
  draggedFromFixed = false; draggedFromError = false;
}}

function toggleComment(key, elem) {{
  event.stopPropagation();
  if (activeElement === elem) {{ closeComment(); }}
  else {{ activeElement = elem; renderDetail(); }}
}}

function showCommentSection(key, elem) {{
  const commentKey = getCommentKey(key, elem);
  const existingComment = comments[commentKey] || '';
  const pills = document.querySelectorAll('.pill');
  let targetPill = null;
  pills.forEach(pill => {{
    if (pill.title === elem || pill.title.startsWith(elem + ' ')) targetPill = pill;
  }});
  if (!targetPill) return;
  const typeBlock = targetPill.closest('.type-block');
  let commentContainer = typeBlock.querySelector('.comment-container');
  if (!commentContainer) {{
    commentContainer = document.createElement('div');
    commentContainer.className = 'comment-container';
    typeBlock.appendChild(commentContainer);
  }}
  const isEditing = !existingComment;
  const btnText   = existingComment ? 'Actualizar' : 'Guardar';
  const deleteBtn = existingComment
    ? `<button class="comment-btn delete" onclick="deleteComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')" >Eliminar</button>`
    : '';
  const editingHTML = `
    <textarea class="comment-textarea" id="commentText" placeholder="Escribe tu comentario aquí...">${{existingComment}}</textarea>
    <div class="comment-actions">
      <button class="comment-btn save" onclick="saveComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')">${{btnText}}</button>
      <button class="comment-btn cancel" onclick="closeComment()">Cancelar</button>
      ${{deleteBtn}}
    </div>`;
  const displayHTML = `
    <div class="comment-display">${{existingComment}}</div>
    <div class="comment-actions">
      <button class="comment-btn save" onclick="editComment()">Editar</button>
      <button class="comment-btn delete" onclick="deleteComment('${{key}}', '${{elem.replace(/'/g, "\\\\'")}}')" >Eliminar</button>
    </div>`;
  commentContainer.innerHTML = `
    <div class="comment-section show">
      <div class="comment-header">
        <span class="elem-name">${{elem}}</span>
        <button class="close-btn" onclick="closeComment()">×</button>
      </div>
      ${{isEditing ? editingHTML : displayHTML}}
    </div>`;
  if (isEditing) {{
    setTimeout(() => {{ const ta = document.getElementById('commentText'); if (ta) ta.focus(); }}, 50);
  }}
}}

function editComment() {{ if (activeKey && activeElement) renderDetail(); }}

function saveComment(key, elem) {{
  const textarea = document.getElementById('commentText');
  if (!textarea) return;
  const comment    = textarea.value.trim();
  const commentKey = getCommentKey(key, elem);
  if (comment) {{ comments[commentKey] = comment; }}
  else         {{ delete comments[commentKey]; }}
  saveComments();
  renderDetail();
}}

function deleteComment(key, elem) {{
  if (!confirm('¿Estás seguro de que quieres eliminar este comentario?')) return;
  const commentKey = getCommentKey(key, elem);
  delete comments[commentKey];
  saveComments();
  activeElement = null;
  renderDetail();
}}

function closeComment() {{ activeElement = null; renderDetail(); }}
 
function selectKey(k) {{
  activeKey     = activeKey === k ? null : k;
  activeElement = null;
  renderKeys();
  renderDetail();
}}
 
renderKeys();
</script>
</body>
</html>"""
    return html

<string>:316: SyntaxWarning: invalid escape sequence '\{'
<string>:316: SyntaxWarning: invalid escape sequence '\}'
<string>:320: SyntaxWarning: invalid escape sequence '\{'
<string>:320: SyntaxWarning: invalid escape sequence '\}'
<string>:324: SyntaxWarning: invalid escape sequence '\{'
<string>:324: SyntaxWarning: invalid escape sequence '\}'
<>:316: SyntaxWarning: invalid escape sequence '\{'
<>:316: SyntaxWarning: invalid escape sequence '\}'
<>:320: SyntaxWarning: invalid escape sequence '\{'
<>:320: SyntaxWarning: invalid escape sequence '\}'
<>:324: SyntaxWarning: invalid escape sequence '\{'
<>:324: SyntaxWarning: invalid escape sequence '\}'
<>:745: SyntaxWarning: invalid escape sequence '\s'
<>:736: SyntaxWarning: invalid escape sequence '\s'
<>:736: SyntaxWarning: invalid escape sequence '\s'
<string>:316: SyntaxWarning: invalid escape sequence '\{'
<string>:316: SyntaxWarning: invalid escape sequence '\}'
<string>:320: SyntaxWarning: invalid escape sequence '\{'
<string>:3

In [43]:
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Documentos\Analisis Catálogo Virtual\2026_03_27_{}.html".format(CTC))

In [44]:
html_content = build_html(
    Topo_sin_recibir,
    recibir_sin_topo,
    label_a='ELEMENTOS EN CATÁLOGO CTC Y EN TOPOLOGÍA MSE VIEW DE LOS QUE NO SE RECIBE INFORMACIÓN (INCIDENCIA DE TECNÓLOGO CTC)',
    label_b='ELEMENTOS PUBLICADOS POR CTC QUE NO ESTÁN CORRECTOS EN TOPOLOGÍA MSE VIEW (INCIDENCIA TOPOLOGÍA MSE)',
    output_title=f'Catálogo virtual CTC: {CTC}'
)
with open(fname, 'w', encoding='utf-8') as f:
    f.write(html_content)
